> **Versión corregida (v2.13, línea v1 de despliegue, ejecución verificada)** — generada a partir de `notebooks/12_generar_dashboard_streamlit.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización: se sustituyen la detección/montaje de Drive y el clonado de `main` por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`, rama `feature/entorno-capacidad-v2`).
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; `PERSISTENT_ROOT` se conserva como raíz de la línea v1 (= `PROJECT_ROOT`, o `PROJECT_ROOT/prueba_linea_v1` en modo prueba).
> - El dashboard v1 se sigue generando en `PERSISTENT_ROOT/dashboard_streamlit/`; al publicarse (notebook 13) se copia a `dashboard_streamlit_v1/` para no sobrescribir el dashboard V2 del repositorio.
> - El README generado indica que, en el repositorio, la aplicación v1 vive en `dashboard_streamlit_v1/app.py`.
> - Nueva sección 9: exporta el reporte integral V2 a `dashboard_data/v2/` (sin `detalle.csv`), regenera el manifiesto y lo valida con el cargador del dashboard.
> - La sección 9 también exporta, si existen, las tablas del notebook 10 (`criterios_pida`, `rendimiento_ic`, `robustez_escenarios`, `brechas_subgrupos_pida`, `verificacion_diccionario`) para la sección «Criterios PIDA» del dashboard V2.
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña, pocos pasos, salida en `prueba_linea_v1/`).


# PIDA-RL Diabetes — 12. Generación del dashboard Streamlit

Este notebook genera una aplicación Streamlit reproducible para explorar las políticas evaluadas. La aplicación carga exclusivamente los archivos compactos creados por el notebook 11; no reentrena modelos ni carga trazas grandes.

> **Uso responsable:** el dashboard presenta resultados de simulación. No es un sistema clínico, no sustituye criterio profesional y no debe emplearse para tomar decisiones reales sobre pacientes.


## 1. Objetivo

Crear y validar `app.py`, una aplicación de Streamlit con:

- Resumen ejecutivo y delimitación de alcance.
- Comparación de las seis políticas.
- Cobertura por nivel de riesgo.
- Estabilidad de cobertura de alto riesgo entre semillas.
- Conclusiones, limitaciones y criterios de interpretación.

La aplicación se guarda en Google Drive para persistir entre sesiones de Colab.


## 2. Inicialización


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'feature/entorno-capacidad-v2')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

# --- Línea v1 de despliegue (entorno autocontenido) ---
# En modo prueba se escribe en una carpeta aparte para no sobrescribir resultados completos ni el dashboard publicado.
LINEA_V1_ROOT = PROJECT_ROOT / 'prueba_linea_v1' if MODO_PRUEBA else PROJECT_ROOT
PERSISTENT_ROOT = LINEA_V1_ROOT
COHORTE_CANONICA = PROJECT_ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
print('Raíz de la línea v1 (PERSISTENT_ROOT):', PERSISTENT_ROOT)

# --- Celda original (sin montaje de Drive ni clonado fijo) ---
from pathlib import Path
import os
import subprocess
import sys
import json
import textwrap

DASHBOARD_DATA_DIR = PERSISTENT_ROOT / 'dashboard_data'
DASHBOARD_DIR = PERSISTENT_ROOT / 'dashboard_streamlit'
APP_PATH = DASHBOARD_DIR / 'app.py'
REQUIREMENTS_PATH = DASHBOARD_DIR / 'requirements.txt'
README_PATH = DASHBOARD_DIR / 'README.md'

for directory in [DASHBOARD_DATA_DIR, DASHBOARD_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ROOT)
print('Datos del dashboard:', DASHBOARD_DATA_DIR)
print('Aplicación Streamlit:', DASHBOARD_DIR)


## 3. Validar datos de entrada


In [ ]:
DATA_FILES = {
    'decision_table': DASHBOARD_DATA_DIR / 'tabla_decision_final_v1.csv',
    'risk_coverage': DASHBOARD_DATA_DIR / 'cobertura_por_riesgo_dashboard_v1.csv',
    'high_risk_stability': DASHBOARD_DATA_DIR / 'alto_riesgo_por_semilla_dashboard_v1.csv',
    'conclusions': DASHBOARD_DATA_DIR / 'conclusiones_dashboard_v1.csv',
    'configuration': DASHBOARD_DATA_DIR / 'configuracion_dashboard_v1.json',
}

for name, path in DATA_FILES.items():
    print(f'{name:22s} | existe: {path.is_file()} | bytes: {path.stat().st_size if path.is_file() else 0}')

missing_files = [str(path) for path in DATA_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Faltan insumos del notebook 11:\n' + '\n'.join(missing_files)
    )

with DATA_FILES['configuration'].open('r', encoding='utf-8') as file:
    dashboard_config = json.load(file)

print('Política seleccionada:', dashboard_config.get('selected_policy'))
print('Estado:', dashboard_config.get('recommendation_status'))


## 4. Definir aplicación

La aplicación utiliza rutas relativas: espera encontrar una carpeta `dashboard_data` junto a `app.py`. Esto permite mover ambos directorios al repositorio de GitHub o a otro entorno sin modificar el código.


In [ ]:
app_code = r'''from pathlib import Path
import json

import pandas as pd
import plotly.express as px
import streamlit as st


st.set_page_config(
    page_title='PIDA-RL Diabetes México',
    page_icon='🩺',
    layout='wide',
    initial_sidebar_state='expanded',
)

APP_DIR = Path(__file__).resolve().parent
DATA_DIR = APP_DIR.parent / 'dashboard_data'

DECISION_PATH = DATA_DIR / 'tabla_decision_final_v1.csv'
RISK_PATH = DATA_DIR / 'cobertura_por_riesgo_dashboard_v1.csv'
STABILITY_PATH = DATA_DIR / 'alto_riesgo_por_semilla_dashboard_v1.csv'
CONCLUSIONS_PATH = DATA_DIR / 'conclusiones_dashboard_v1.csv'
CONFIG_PATH = DATA_DIR / 'configuracion_dashboard_v1.json'


@st.cache_data(show_spinner=False)
def load_data():
    required_paths = [DECISION_PATH, RISK_PATH, STABILITY_PATH, CONCLUSIONS_PATH, CONFIG_PATH]
    missing = [str(path) for path in required_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError('No se encontraron estos archivos requeridos:\n' + '\n'.join(missing))

    decision = pd.read_csv(DECISION_PATH)
    risk = pd.read_csv(RISK_PATH)
    stability = pd.read_csv(STABILITY_PATH)
    conclusions = pd.read_csv(CONCLUSIONS_PATH)
    with CONFIG_PATH.open('r', encoding='utf-8') as file:
        config = json.load(file)

    return decision, risk, stability, conclusions, config


def format_boolean(value):
    return 'Sí' if bool(value) else 'No'


def policy_label(policy):
    labels = {
        'aleatoria': 'Aleatoria',
        'reglas_riesgo': 'Reglas de riesgo',
        'reglas_capacidad': 'Reglas con capacidad',
        'dqn_v1': 'DQN',
        'q_learning_v1': 'Q-learning',
        'ppo_v1': 'PPO',
    }
    return labels.get(policy, policy)


def add_labels(frame):
    output = frame.copy()
    if 'politica' in output.columns:
        output['Política'] = output['politica'].map(policy_label)
    return output


def main():
    st.title('Priorización adaptativa de seguimiento en diabetes')
    st.caption('PIDA-RL México | Comparación de políticas en un entorno de simulación')

    st.warning(
        'Resultados de simulación con datos públicos y supuestos explícitos. '
        'Esta aplicación no es un dispositivo médico, no usa datos de pacientes reales '
        'y no debe emplearse para decisiones clínicas individuales.'
    )

    try:
        decision, risk, stability, conclusions, config = load_data()
    except Exception as error:
        st.error(f'No fue posible cargar los datos del dashboard: {error}')
        st.stop()

    selected_policy = config.get('selected_policy', 'No disponible')
    recommendation_status = config.get('recommendation_status', 'No disponible')

    decision = add_labels(decision)
    risk = add_labels(risk)
    stability = add_labels(stability)

    st.sidebar.header('Navegación')
    section = st.sidebar.radio(
        'Selecciona una sección',
        ['Resumen ejecutivo', 'Comparación de políticas', 'Cobertura por riesgo', 'Estabilidad', 'Conclusiones y límites'],
    )

    st.sidebar.divider()
    st.sidebar.subheader('Política de referencia')
    st.sidebar.write(policy_label(selected_policy))
    st.sidebar.caption(recommendation_status.replace('_', ' '))

    if section == 'Resumen ejecutivo':
        st.header('Resumen ejecutivo')
        best = decision.loc[decision['politica'] == selected_policy]
        if best.empty:
            st.info('La política seleccionada no aparece en la tabla de decisión.')
        else:
            best = best.iloc[0]
            col1, col2, col3, col4 = st.columns(4)
            col1.metric('Política de referencia', policy_label(selected_policy))
            col2.metric('Ranking técnico', int(best['ranking_tecnico']))
            col3.metric('Puntaje técnico', f"{best['puntaje_tecnico']:.3f}")
            col4.metric('Cobertura alto riesgo', f"{best['alto']:.2f}%")

            st.subheader('Interpretación')
            st.write(
                'La política de referencia fue seleccionada bajo una regla técnica reproducible: '
                'primer lugar del ranking, respeto promedio de capacidad y cobertura positiva de alto riesgo. '
                'La selección solo aplica al entorno de simulación.'
            )

        st.subheader('Cobertura y capacidad')
        overview_columns = [
            'Política',
            'ranking_tecnico',
            'puntaje_tecnico',
            'alto',
            'excesos_capacidad_medios',
            'respeta_capacidad_en_promedio',
        ]
        overview = decision[[column for column in overview_columns if column in decision.columns]].copy()
        if 'respeta_capacidad_en_promedio' in overview.columns:
            overview['respeta_capacidad_en_promedio'] = overview['respeta_capacidad_en_promedio'].map(format_boolean)
        st.dataframe(overview, use_container_width=True, hide_index=True)

    elif section == 'Comparación de políticas':
        st.header('Comparación de políticas')
        visible_columns = [
            'Política',
            'ranking_tecnico',
            'clasificacion',
            'puntaje_tecnico',
            'recompensa_media',
            'cobertura_alto_riesgo_media',
            'bajo',
            'medio',
            'alto',
            'eventos_adversos_medios',
            'recursos_usados_medios',
            'excesos_capacidad_medios',
            'respeta_capacidad_en_promedio',
        ]
        comparison = decision[[column for column in visible_columns if column in decision.columns]].copy()
        if 'respeta_capacidad_en_promedio' in comparison.columns:
            comparison['respeta_capacidad_en_promedio'] = comparison['respeta_capacidad_en_promedio'].map(format_boolean)
        st.dataframe(comparison, use_container_width=True, hide_index=True)

        chart = px.bar(
            decision.sort_values('ranking_tecnico'),
            x='Política',
            y='puntaje_tecnico',
            color='clasificacion',
            text='puntaje_tecnico',
            title='Puntaje técnico por política',
            labels={'puntaje_tecnico': 'Puntaje técnico', 'clasificacion': 'Tipo de política'},
        )
        chart.update_traces(texttemplate='%{text:.3f}', textposition='outside')
        chart.update_layout(yaxis_range=[0, max(1.0, decision['puntaje_tecnico'].max() * 1.15)])
        st.plotly_chart(chart, use_container_width=True)

        capacity_chart = px.bar(
            decision.sort_values('ranking_tecnico'),
            x='Política',
            y='excesos_capacidad_medios',
            color='respeta_capacidad_en_promedio',
            title='Excesos promedio de capacidad',
            labels={
                'excesos_capacidad_medios': 'Excesos por episodio',
                'respeta_capacidad_en_promedio': 'Respeta capacidad',
            },
        )
        st.plotly_chart(capacity_chart, use_container_width=True)

    elif section == 'Cobertura por riesgo':
        st.header('Cobertura por nivel de riesgo')
        st.write(
            'La cobertura indica el porcentaje de decisiones agregadas por paso en las que el contacto fue aceptado. '
            'En una estrategia de priorización, una cobertura mayor de alto riesgo puede ser coherente con el objetivo, '
            'siempre que los otros niveles no queden sin atención de manera extrema.'
        )

        policy_options = decision.sort_values('ranking_tecnico')['politica'].tolist()
        selected_policies = st.multiselect(
            'Políticas a visualizar',
            options=policy_options,
            default=policy_options,
            format_func=policy_label,
        )
        filtered_risk = risk[risk['politica'].isin(selected_policies)].copy()

        risk_chart = px.bar(
            filtered_risk,
            x='Política',
            y='cobertura_media',
            color='categoria_riesgo',
            barmode='group',
            title='Cobertura media de seguimiento por nivel de riesgo',
            labels={
                'cobertura_media': 'Cobertura media (%)',
                'categoria_riesgo': 'Nivel de riesgo',
            },
            category_orders={'categoria_riesgo': ['bajo', 'medio', 'alto']},
            color_discrete_map={'bajo': '#72B7B2', 'medio': '#F2CF5B', 'alto': '#E45756'},
        )
        st.plotly_chart(risk_chart, use_container_width=True)

        risk_columns = [
            'Política',
            'categoria_riesgo',
            'cobertura_media',
            'cobertura_desviacion',
            'contactos_medios',
            'tasa_eventos_media',
            'costo_medio',
            'excesos_capacidad_medios',
        ]
        st.dataframe(
            filtered_risk[[column for column in risk_columns if column in filtered_risk.columns]],
            use_container_width=True,
            hide_index=True,
        )

    elif section == 'Estabilidad':
        st.header('Estabilidad de cobertura de alto riesgo')
        st.write(
            'La gráfica compara la cobertura de alto riesgo entre las semillas de evaluación. '
            'Diferencias grandes entre semillas sugieren sensibilidad a la inicialización o a la dinámica estocástica del simulador.'
        )

        stability_chart = px.line(
            stability,
            x='semilla',
            y='cobertura_porcentaje',
            color='Política',
            markers=True,
            title='Cobertura de alto riesgo por semilla',
            labels={'semilla': 'Semilla', 'cobertura_porcentaje': 'Cobertura alto riesgo (%)'},
        )
        st.plotly_chart(stability_chart, use_container_width=True)

        stability_columns = [
            'Política',
            'semilla',
            'decisiones',
            'contactos_aceptados',
            'cobertura_porcentaje',
            'eventos_adversos',
            'costo_total',
            'excesos_capacidad',
        ]
        st.dataframe(
            stability[[column for column in stability_columns if column in stability.columns]],
            use_container_width=True,
            hide_index=True,
        )

    else:
        st.header('Conclusiones y límites')
        for _, row in conclusions.iterrows():
            label = str(row.get('tipo', 'resultado')).replace('_', ' ').capitalize()
            st.subheader(label)
            st.write(row.get('contenido', ''))

        st.subheader('Limitaciones de la auditoría')
        st.info(
            'La auditoría disponible compara resultados por nivel de riesgo. '
            'No se presentan resultados por sexo ni edad porque las trazas agregadas de evaluación no incluyen '
            'atributos demográficos ni identificadores individuales.'
        )

        st.subheader('Requisitos previos a cualquier uso real')
        st.markdown(
            '- Datos institucionales autorizados y gobernanza de datos.\n'
            '- Validación clínica y evaluación prospectiva.\n'
            '- Auditoría de desempeño y equidad con subgrupos demográficos observables.\n'
            '- Revisión ética, trazabilidad, monitoreo y supervisión humana.'
        )


if __name__ == '__main__':
    main()
'''

APP_PATH.write_text(app_code, encoding='utf-8')
print('Aplicación generada:', APP_PATH)
print('Bytes:', APP_PATH.stat().st_size)


## 5. Crear archivos auxiliares


In [ ]:
requirements_content = '''streamlit>=1.36
pandas>=2.0
plotly>=5.20
'''
REQUIREMENTS_PATH.write_text(requirements_content, encoding='utf-8')

readme_content = '''# Dashboard PIDA-RL Diabetes México

## Propósito

Aplicación Streamlit para explorar resultados de simulación de políticas de priorización de seguimiento en personas con diabetes.

## Ejecución local

1. Instala dependencias:

```bash
pip install -r requirements.txt
```

2. Mantén la carpeta `dashboard_data` junto a la carpeta `dashboard_streamlit`.

3. Desde el directorio que contiene ambas carpetas ejecuta:

```bash
streamlit run dashboard_streamlit/app.py
```

En el repositorio GitHub esta aplicación se publica como `dashboard_streamlit_v1/app.py` (la carpeta `dashboard_streamlit/` contiene el dashboard principal V2).

## Alcance

Los resultados provienen de un simulador con datos públicos y supuestos explícitos. La aplicación no usa datos clínicos reales, no es un dispositivo médico y no debe utilizarse para decisiones clínicas individuales.

La auditoría disponible es por nivel de riesgo. No se infieren ni se muestran resultados por sexo o edad porque las trazas de evaluación agregadas no contienen esos atributos.
'''
README_PATH.write_text(readme_content, encoding='utf-8')

for path in [REQUIREMENTS_PATH, README_PATH]:
    print(path.name, '| existe:', path.is_file(), '| bytes:', path.stat().st_size if path.is_file() else 0)


## 6. Validación sintáctica

Esta validación compila `app.py` sin iniciar el servidor. Detecta errores de sintaxis antes de intentar ejecutar Streamlit.


In [ ]:
import py_compile

py_compile.compile(str(APP_PATH), doraise=True)
print('Validación sintáctica aprobada: app.py compila correctamente.')


## 7. Vista previa del código


In [ ]:
print(APP_PATH.read_text(encoding='utf-8')[:5000])


## 8. Instrucciones de ejecución

Para ejecutar la aplicación localmente desde una terminal ubicada dentro de `PIDA-RL-Diabetes`, usa:

```bash
pip install -r dashboard_streamlit/requirements.txt
streamlit run dashboard_streamlit/app.py
```

En Google Colab, puede instalarse Streamlit y exponerse temporalmente mediante un túnel. Esa publicación es efímera y no debe emplearse con datos sensibles.


In [ ]:
dashboard_metadata = {
    'version': '1.0',
    'generated_at_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),
    'app_path': str(APP_PATH),
    'data_directory': str(DASHBOARD_DATA_DIR),
    'required_data_files': {name: str(path) for name, path in DATA_FILES.items()},
    'selected_policy': dashboard_config.get('selected_policy'),
    'recommendation_status': dashboard_config.get('recommendation_status'),
    'scope_note': 'Dashboard para resultados de simulación; no usar en decisiones clínicas reales.',
}

metadata_path = DASHBOARD_DIR / 'dashboard_build_metadata_v1.json'
with metadata_path.open('w', encoding='utf-8') as file:
    json.dump(dashboard_metadata, file, ensure_ascii=False, indent=2)

for path in [APP_PATH, REQUIREMENTS_PATH, README_PATH, metadata_path]:
    print(path.name, '| existe:', path.is_file(), '| bytes:', path.stat().st_size if path.is_file() else 0)

print('Dashboard Streamlit generado y validado correctamente.')


## 9. Exportar datos del dashboard principal (V2)

El dashboard principal del repositorio (`dashboard_streamlit/app.py`) consume el reporte integral V2. Esta celda copia solo las tablas agregadas a `dashboard_data/v2/`, regenera `manifest.csv` y valida el conjunto con `data_loader.load_integral_report`.


In [ ]:
# Exporta el reporte integral V2 (notebook 09_evaluacion_integral_equidad_v2) para el dashboard principal.
# Se publican solo tablas agregadas: se excluye detalle.csv (traza por perfil y mes, voluminosa).
import shutil
import pandas as pd

V2_REPORT_DIR = RESULTS_V2_DIR / 'evaluacion_integral'
V2_DASHBOARD_DIR = DASHBOARD_DATA_DIR / 'v2'
TABLAS_V2 = [
    'episodios', 'resumen_algoritmos', 'clasificaciones_propuestas', 'clasificaciones_ejecutadas',
    'estabilidad_semillas', 'metricas_subgrupos', 'brechas_equidad',
    'sensibilidad_episodios', 'sensibilidad_resumen',
]
V2_EXPORTADO = False
faltan_v2 = [t for t in TABLAS_V2[:7] if not (V2_REPORT_DIR / f'{t}.csv').is_file()]
if faltan_v2:
    print('ADVERTENCIA: no se encontró el reporte integral V2 completo en', V2_REPORT_DIR)
    print('Ejecuta los notebooks V2 05–09 para habilitar el dashboard principal. Faltan:', faltan_v2)
else:
    V2_DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
    filas = []
    for t in TABLAS_V2:
        origen = V2_REPORT_DIR / f'{t}.csv'
        if origen.is_file():
            shutil.copy2(origen, V2_DASHBOARD_DIR / origen.name)
            filas.append({'tabla': t, 'archivo': origen.name, 'filas': len(pd.read_csv(origen))})
    # Tablas del notebook 10 (cumplimiento de la Definición del PIDA), si ya se ejecutó.
    PIDA_DIR = RESULTS_V2_DIR / 'cumplimiento_pida'
    for t in ['criterios_pida', 'rendimiento_ic', 'robustez_escenarios', 'brechas_subgrupos_pida', 'verificacion_diccionario']:
        origen = PIDA_DIR / f'{t}.csv'
        if origen.is_file():
            shutil.copy2(origen, V2_DASHBOARD_DIR / origen.name)
            filas.append({'tabla': t, 'archivo': origen.name, 'filas': len(pd.read_csv(origen))})
        else:
            print('Aviso: falta', origen.name, '(ejecuta 10_cumplimiento_pida_v2 para la sección «Criterios PIDA»).')
    obsoleto = V2_DASHBOARD_DIR / 'detalle.csv'
    if obsoleto.exists():
        obsoleto.unlink()
    pd.DataFrame(filas).to_csv(V2_DASHBOARD_DIR / 'manifest.csv', index=False)

    # Validación con el mismo cargador que usa la aplicación desplegada.
    sys.path.insert(0, str(REPO_ROOT / 'dashboard_streamlit'))
    from data_loader import load_integral_report
    reporte_v2 = load_integral_report(V2_DASHBOARD_DIR)
    print('Reporte V2 exportado y validado:', V2_DASHBOARD_DIR)
    print('Huella del reporte:', reporte_v2['fingerprint'])
    display(pd.DataFrame(filas))
    V2_EXPORTADO = True


## Resultado esperado

El notebook debe finalizar con:

```text
Dashboard Streamlit generado y validado correctamente.
```

Los archivos persistentes estarán en `PIDA-RL-Diabetes/dashboard_streamlit/`: `app.py`, `requirements.txt`, `README.md` y `dashboard_build_metadata_v1.json`.
